In [1]:
pip install mem0ai langgraph langchain-openai python-dotenv chromadb dashscope

  Using cached dashscope-1.24.4-py3-none-any.whl.metadata (7.1 kB)
Using cached dashscope-1.24.4-py3-none-any.whl (1.3 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

os.environ["OPENAI_API_KEY"] = "sk-**********************"
os.environ["OPENAI_BASE_URL"] = "https://dashscope.aliyuncs.com/compatible-mode/v1"
os.environ["OPENAI_MODEL"] = "qwen-turbo"
os.environ["TAVILY_API_KEY"] = "tvly-3*****************"

In [3]:
import os
from dotenv import load_dotenv
from typing import Annotated, List, TypedDict

from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

from mem0 import Memory
from mem0.configs.base import MemoryConfig
from mem0.llms.configs import LlmConfig
from mem0.embeddings.configs import EmbedderConfig
from langchain_community.embeddings import DashScopeEmbeddings

# 加载环境变量
load_dotenv()

# 1. 配置 LLM
# temperature=0 确保输出更具确定性
llm = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
    temperature=0.2,
    timeout=1200,
    max_retries=2,
)

# 2. 配置 Mem0
# Mem0 需要一个 LLM 来进行记忆的推断和管理，
# 以及一个 Embedding 模型来将记忆向量化以便检索。
# 这里我们都使用 OpenAI 的服务。
# 这是一个非常关键的配置步骤。
dashscope_embeddings = DashScopeEmbeddings(
    model="text-embedding-v3",
    dashscope_api_key=os.getenv("OPENAI_API_KEY"),
)

mem0_config = {
    "llm": {
        "provider": "openai",
        "config": {
            "model": os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            "temperature": 0,
            "api_key": os.getenv("OPENAI_API_KEY"),
            "openai_base_url": os.getenv("OPENAI_BASE_URL"),
        },
    },
    "embedder": {
        "provider": "langchain",
        "config": {"model": dashscope_embeddings}
    },
    "vector_store": {
        "provider": "chroma", # 使用本地的 ChromaDB 作为向量存储
        "config": {
            "path": "./mem0_db" # 数据库文件将保存在这个路径
        }
    }
}

# 初始化 Mem0 实例
# 这个 memory_instance 就是我们与记忆层交互的唯一入口
memory_instance = Memory.from_config(mem0_config)

print("环境配置完成，LLM 和 Mem0 已成功初始化。")

环境配置完成，LLM 和 Mem0 已成功初始化。


In [4]:
# 3. 定义 LangGraph 的状态
class AgentState(TypedDict):
    """
    定义 Agent 在图中流转的状态
    """
    # `add_messages` 是一个辅助函数，它能方便地将新消息追加到列表中
    messages: Annotated[List[BaseMessage], add_messages]
    
    # 用于区分不同用户的记忆
    mem0_user_id: str

print("Agent 状态定义完成。")

Agent 状态定义完成。


In [7]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

def _normalize_mem_texts(items):
    """将 mem0.search 的结果统一转成字符串列表，兼容多种 schema。"""
    texts = []
    for m in (items or []):
        if isinstance(m, str):
            texts.append(m)
        elif isinstance(m, dict):
            # 常见键位的优先级尝试
            for k in ("memory", "text", "content", "value", "data", "summary"):
                v = m.get(k)
                if isinstance(v, str) and v.strip():
                    texts.append(v)
                    break
            else:
                # 都没有匹配上，兜底转字符串
                texts.append(str(m))
        else:
            texts.append(str(m))
    return texts

def travel_agent_node(state: AgentState):
    """
    这是我们 Agent 的核心逻辑节点。
    它执行“检索-注入-生成-存储”的记忆循环。
    """
    print("\n--- 进入 Travel Agent 节点 ---")

    # 从状态中获取当前的用户ID和最新的用户消息
    user_id = state["mem0_user_id"]
    last_user_message = state["messages"][-1].content
    
    # --- 步骤 1: 检索记忆 (Retrieve) ---
    print(f"正在为用户 '{user_id}' 检索关于 '{last_user_message}' 的记忆...")
    relevant_memories = memory_instance.search(query=last_user_message, user_id=user_id)
    # 关键：把返回做归一化，避免 'mem["memory"]' 报错
    memory_texts = _normalize_mem_texts(relevant_memories)

    # --- 步骤 2: 注入上下文 (Inject) ---
    system_prompt_template = (
        "You are a helpful and personalized travel agent. "
        "Use the provided context from past conversations to offer tailored recommendations. "
        "If no context is provided, act as a general travel agent."
    )

    memory_context = ""
    if memory_texts:
        joined_memory = "\n- ".join(memory_texts)
        memory_context = "\n\nRelevant context from our past conversations:\n- " + joined_memory
        print("检索到相关记忆:\n- " + joined_memory)
    else:
        print("未检索到相关记忆。")

    full_system_prompt = system_prompt_template + memory_context

    # --- 步骤 3: 生成回复 (Generate) ---
    print("调用 LLM 生成回复...")
    messages_for_llm = [
        SystemMessage(content=full_system_prompt),  # 用 SystemMessage 更语义化
        *state["messages"]
    ]
    response = llm.invoke(messages_for_llm)
    print(f"LLM 回复: {response.content}")

    # --- 步骤 4: 存储新记忆 (Store) ---
    interaction_to_save = f"User: {last_user_message}\nAssistant: {response.content}"
    print("正在将本轮交互存入记忆库...")
    memory_instance.add(interaction_to_save, user_id=user_id)
    print("记忆存储完成。")

    # 返回对状态的更新
    return {"messages": [response]}

In [9]:
# 5. 构建并编译 LangGraph
# 初始化一个状态图，并指定其状态结构
graph_builder = StateGraph(AgentState)

# 将我们的核心逻辑函数添加为图中的一个节点，命名为 "travel_agent"
graph_builder.add_node("travel_agent", travel_agent_node)

# 设置图的入口点，即程序开始时首先执行哪个节点
graph_builder.set_entry_point("travel_agent")

# 添加一条边，让 "travel_agent" 节点执行完毕后，流程结束
# 在更复杂的 Agent 中，这里可以添加条件边，导向其他节点（如工具调用）
graph_builder.add_edge("travel_agent", END)

# 编译图，得到一个可执行的 app 对象
app = graph_builder.compile()

print("LangGraph 构建并编译完成。Agent 已准备就绪！")

# 6. 运行对话，见证记忆的效果
def run_conversation(user_id: str):
    """
    一个简单的函数来模拟与 Agent 的连续对话
    """
    # 为每个用户维护一个独立的对话历史
    # 在真实应用中，这部分状态管理会更复杂
    messages = []
    
    while True:
        user_input = input(f"You ({user_id}): ")
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("Agent: 感谢使用，祝您旅途愉快！")
            break
        
        # 将用户输入添加到对话历史中
        messages.append(HumanMessage(content=user_input))
        
        # 调用我们编译好的 LangGraph app
        # 传入当前的状态
        state = {"messages": messages, "mem0_user_id": user_id}
        result = app.invoke(state)
        
        # 从返回结果中获取最新的 AI 回复，并更新对话历史
        ai_response = result["messages"][-1]
        messages.append(ai_response)
        
        print(f"Agent: {ai_response.content}")

# --- 演示开始 ---
if __name__ == "__main__":
    print("\n--- 欢迎来到个性化旅行助手 ---")
    print("您可以随时输入 'quit', 'exit' 或 'bye' 来结束对话。")
    
    # 模拟与用户 "alice" 的对话
    # 您可以更改 user_id 来体验不同用户的记忆隔离
    current_user_id = "alice" 
    run_conversation(user_id=current_user_id)

LangGraph 构建并编译完成。Agent 已准备就绪！

--- 欢迎来到个性化旅行助手 ---
您可以随时输入 'quit', 'exit' 或 'bye' 来结束对话。


You (alice):  嗨，我又来了，帮我推荐一个周末旅行的好去处吧。



--- 进入 Travel Agent 节点 ---
正在为用户 'alice' 检索关于 '嗨，我又来了，帮我推荐一个周末旅行的好去处吧。' 的记忆...
检索到相关记忆:
- results
调用 LLM 生成回复...
LLM 回复: 嗨！很高兴再次见到你！周末旅行的话，我们可以考虑一些短途但风景优美的地方。根据你之前提到的兴趣（比如自然风光、文化体验或者美食），我有几个推荐：

1. **杭州** - 如果你喜欢自然和文化，西湖边的景色非常迷人，还可以品尝地道的杭帮菜。
2. **苏州** - 有美丽的园林和古色古香的街道，适合喜欢历史和文化的你。
3. **南京** - 有丰富的历史遗迹，比如中山陵、夫子庙，还有美味的南京小吃。
4. **上海周边小镇** - 比如朱家角或西塘，可以体验江南水乡的宁静与美丽。

你更倾向于哪种类型的旅行呢？我可以根据你的偏好进一步推荐哦！
正在将本轮交互存入记忆库...
记忆存储完成。
Agent: 嗨！很高兴再次见到你！周末旅行的话，我们可以考虑一些短途但风景优美的地方。根据你之前提到的兴趣（比如自然风光、文化体验或者美食），我有几个推荐：

1. **杭州** - 如果你喜欢自然和文化，西湖边的景色非常迷人，还可以品尝地道的杭帮菜。
2. **苏州** - 有美丽的园林和古色古香的街道，适合喜欢历史和文化的你。
3. **南京** - 有丰富的历史遗迹，比如中山陵、夫子庙，还有美味的南京小吃。
4. **上海周边小镇** - 比如朱家角或西塘，可以体验江南水乡的宁静与美丽。

你更倾向于哪种类型的旅行呢？我可以根据你的偏好进一步推荐哦！


You (alice):  quit


Agent: 感谢使用，祝您旅途愉快！
